# v5 - Classification head com MedSigLIP + Derm7pt + HAM10000

Versao v5: mescla Derm7pt + HAM10000 para aumentar drasticamente o numero de amostras das classes minoritarias (BCC, MEL, SK).

**Schema metadata unificado (12 dim) - apenas features compartilhadas:**
- `sex` (3): female, male, unknown
- `location` (9): abdomen, back, head_neck, upper_limbs, lower_limbs, acral, chest, genital, unknown
- *Sem `elevation` ou `age`* — evita domain bias

**Distribuicao de train combinado (~10.000 samples):**
- NEV: 256 (Derm7pt) + ~6000 (HAM) = ~6256
- MEL: 90 + ~1000 = ~1090
- BCC: 19 + ~460 = ~480
- SK: 16 + ~990 = ~1006
- MISC: 32 + ~530 = ~562

**Test:** APENAS Derm7pt test (395 amostras) — comparacao direta com v3/v4.

In [ ]:
import os
import sys
import site
import importlib
import subprocess

REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
site.main()

print("Setup OK")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from collections import Counter
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, CombinedDermDataset,
    Derm7ptClassificationDataset, classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM_V5,
    ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7-pt-dataset/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
DERM_TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
DERM_VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
DERM_TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"

HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Derm7pt meta existe: {os.path.exists(DERM_META)}")
print(f"HAM meta existe: {os.path.exists(HAM_META)}")
print(f"Metadata dim v5: {METADATA_DIM_V5}")

In [ ]:
model, processor = build_dermclassifier(
    hf_token=HF_TOKEN,
    num_classes=5,
    metadata_dim=METADATA_DIM_V5,
    freeze_vision=True,
)
model = model.to(device)
model.vision_encoder = model.vision_encoder.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Derm7pt: usa schema unificado (sem balance, sem augment para o val/test)
derm_train = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                   indexes_csv=DERM_TRAIN_IDX, augment=True, seed=42)
derm_val = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                 indexes_csv=DERM_VAL_IDX, augment=False)
derm_test = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                  indexes_csv=DERM_TEST_IDX, augment=False)

# HAM10000: split por lesion_id (evita leakage paciente)
ham_train_idx, ham_val_idx = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42)
ham_train = HAM10000Dataset(HAM_META, HAM_DIR, processor,
                            indexes=ham_train_idx, augment=True, seed=42)
ham_val = HAM10000Dataset(HAM_META, HAM_DIR, processor,
                          indexes=ham_val_idx, augment=False)

# Combina train e val
train_dataset = CombinedDermDataset([derm_train, ham_train])
val_dataset = CombinedDermDataset([derm_val, ham_val])
test_dataset = derm_test  # SO Derm7pt no test

print(f"Train combinado: {len(train_dataset)} = derm({len(derm_train)}) + ham({len(ham_train)})")
print(f"Val combinado: {len(val_dataset)} = derm({len(derm_val)}) + ham({len(ham_val)})")
print(f"Test (so Derm7pt): {len(test_dataset)}")

# Distribuicao do train (amostra primeiros 1000 + ultimos 500 pra evitar percorrer 10k)
print("\nColetando labels do train (pode demorar uns segundos)...")
train_labels = []
for i in range(0, len(train_dataset), max(1, len(train_dataset)//2000)):
    train_labels.append(train_dataset[i]['labels'].item())
print(f"Sample train distribuicao (n={len(train_labels)}): {Counter([LABEL_TO_GROUP[l] for l in train_labels])}")

test_labels = [test_dataset[i]['labels'].item() for i in range(len(test_dataset))]
print(f"\nTest distribuicao: {Counter([LABEL_TO_GROUP[l] for l in test_labels])}")

In [ ]:
BATCH_SIZE = 32
EPOCHS = 8
LR = 5e-4
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=classification_collate_fn, num_workers=4, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=classification_collate_fn, num_workers=4, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=classification_collate_fn, num_workers=4, pin_memory=True,
)

# Class counts reais do train (precisos)
print("Calculando class counts reais do train...")
real_train_labels = []
for ds in [derm_train, ham_train]:
    for i in range(len(ds)):
        real_train_labels.append(ds[i]['labels'].item())
label_counter = Counter(real_train_labels)
class_counts = [label_counter.get(i, 1) for i in range(5)]
alpha = compute_class_weights(class_counts, mode="inverse_sqrt")
print(f"Class counts REAIS: {class_counts}")
print(f"Class weights (alpha, mode=inverse_sqrt): {alpha.tolist()}")

criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            md = batch['metadata'].to(device)
            lb = batch['labels'].to(device)
            logits = model(pv, md)
            loss = criterion(logits, lb)
            total_loss += loss.item() * lb.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(lb.cpu().tolist())
    n = len(loader.dataset)
    avg_loss = total_loss / n
    acc = sum(int(p==l) for p,l in zip(all_preds, all_labels)) / n
    return avg_loss, acc, all_preds, all_labels

best_val_acc = 0.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    if hasattr(model, 'vision_encoder'):
        model.vision_encoder.eval()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    for batch_idx, batch in enumerate(train_loader):
        pv = batch['pixel_values'].to(device)
        md = batch['metadata'].to(device)
        lb = batch['labels'].to(device)
        logits = model(pv, md)
        loss = criterion(logits, lb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * lb.size(0)
        train_correct += (logits.argmax(dim=-1) == lb).sum().item()
        train_total += lb.size(0)
        if batch_idx % 50 == 0:
            print(f"  Epoch {epoch} batch {batch_idx}/{len(train_loader)} train_loss={loss.item():.4f}")
    scheduler.step()
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} | val_loss={val_loss:.4f} val_acc={val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if 'vision_encoder' not in k}

print(f"\nBest val_acc: {best_val_acc:.3f}")

In [ ]:
if best_state is not None:
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current, strict=False)
    print("Best model carregado.")

os.makedirs("/kaggle/working/derm-classifier-v5", exist_ok=True)
torch.save(best_state, "/kaggle/working/derm-classifier-v5/classifier_head.pt")
print("Salvou /kaggle/working/derm-classifier-v5/classifier_head.pt")

import json
with open("/kaggle/working/derm-classifier-v5/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
test_loss, test_acc, test_preds, test_labels_eval = evaluate(model, test_loader, criterion)
print(f"Test loss (so Derm7pt): {test_loss:.4f}")
print(f"Test accuracy (so Derm7pt): {test_acc:.4f}")

TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]
results = compute_metrics(test_labels_eval, test_preds, target_names=TARGET_NAMES)
plot_confusion_matrix(test_labels_eval, test_preds,
                     save_path='/kaggle/working/derm-classifier-v5-cm.png',
                     target_names=TARGET_NAMES)

In [ ]:
import pandas as pd

test_df = pd.DataFrame({
    'true_label': test_labels_eval,
    'pred_label': test_preds,
    'true_group': [LABEL_TO_GROUP[l] for l in test_labels_eval],
    'pred_group': [LABEL_TO_GROUP[p] for p in test_preds],
})
test_df.to_csv('/kaggle/working/derm_classifier_v5_predictions.csv', index=False)
print(f"Salvas {len(test_df)} predicoes.")
print(f"\nDistribuicao predicoes: {Counter(test_df['pred_group'])}")
print(f"Distribuicao verdadeira: {Counter(test_df['true_group'])}")